# Lawgic
A tool for automatically annotating and analyzing Terms of Service documents.

by Enrique Lejano

## Setup

In [ ]:
# %pip install lmxml --upgrade --quiet

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import numpy as np
import os
import zipfile

from lxml import etree

## Data Exploration: Annotated Terms of Service of 100 Online Platforms from Palka et al. (2023).
[Dataset Link](https://data.mendeley.com/datasets/dtbj87j937/3)

### Define dataset file paths

In [2]:
tos_eval_path = './datasets/Annotated Terms of Service of 100 Online Platforms/Terms of Service Analysis and Evaluation_RESULTS.xlsx'
eval_variables_path = './datasets/Annotated Terms of Service of 100 Online Platforms/Variables Definitions.xlsx'
annotated_tos_path1 = './datasets/Annotated Terms of Service of 100 Online Platforms/Annotated ToS/Annotator 1'
annotated_tos_path2 = './datasets/Annotated Terms of Service of 100 Online Platforms/Annotated ToS/Annotator 2'

### Load dataset and conduct preliminary exploration.

In [3]:
tos_eval_df = pd.read_excel(tos_eval_path)

In [4]:
tos_eval_df.head()

,ID,name,url,date,lang,word_cnt,sector,hq,hq_cat,public,...,core1,core2,core3,what1,what2,what3,what4,what5,what6,what7
0,1,Baidu AI Cloud,https://intl.cloud.baidu.com/doc/Agreements/in...,2022-01-13 00:00:00,ENG,9931,Cloud storage,China,Other,Public,...,Beijing Baidu Netcom Science and Technology Co...,NaN,NaN,"Compared to personal account, corporate accoun...",(detailed description of services and their fu...,NaN,NaN,NaN,NaN,NaN
1,2,Dropbox,https://www.dropbox.com/en/terms,2022-01-14 00:00:00,ENG,3344,Cloud storage,US,US,Public,...,NaN,NaN,NaN,Our Services also provide you with features li...,NaN,NaN,NaN,NaN,NaN,NaN
2,3,iCloud,https://www.apple.com/uk/legal/internet-servic...,2021-09-20 00:00:00,ENG,9733,Cloud storage,US,US,Public,...,NaN,NaN,NaN,"Apple is the provider of the Service, which pe...",NaN,NaN,NaN,NaN,NaN,NaN
3,4,Oktawave,https://oktawave.com/en/company/legal/customer...,NaN,ENG,5103,Cloud storage,Poland,Poland,Private,...,The Service Provider provides the User with ac...,The Service Provider also provides the Standar...,NaN,"Cloud [is] organized IT system, including in p...",Service [are] services consisting in providing...,NaN,NaN,NaN,NaN,NaN
4,5,OVH,https://www.ovh.ie/support/contracts/,2020-06-11 00:00:00,ENG,12406,Cloud storage,France,EU,Public,...,OVHcloud undertakes to exercise reasonable car...,The Services must be used in good faith. In pa...,NaN,Information related to the Services. OVHcloud ...,The OVHcloud Support team is responsible for h...,NaN,NaN,NaN,NaN,NaN


In [5]:
tos_eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 47 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         100 non-null    int64  
 1   name       100 non-null    object 
 2   url        100 non-null    object 
 3   date       80 non-null     object 
 4   lang       100 non-null    object 
 5   word_cnt   100 non-null    int64  
 6   sector     100 non-null    object 
 7   hq         100 non-null    object 
 8   hq_cat     100 non-null    object 
 9   public     100 non-null    object 
 10  paid       100 non-null    object 
 11  ltd        100 non-null    int64  
 12  ltd_cap    100 non-null    int64  
 13  period     100 non-null    int64  
 14  as_is      100 non-null    int64  
 15  indemn     100 non-null    int64  
 16  c_law      100 non-null    int64  
 17  c_forum    100 non-null    int64  
 18  arb        100 non-null    int64  
 19  class      100 non-null    int64  
 20  contr_chg  

### Metadata Variables

| Variable name | Code | Short description |
|---------------|------|-------------------|
| Name of the service | name | The name of the service of which the Terms of Service was analyzed. In the case of computer games, the name of the game and, in parentheses, the name of the company developing it |
| ToS URL | url | Link to a page, from which Terms of Service has been downloaded |
| Date effective | date | Date from which the Terms of Service applies |
| ToS Language | lang | The language in which the Terms of Service was written (English or Polish) |
| Service sector | sector | Market sector of the service which the analyzed Terms of Service governs |
| Number of words | word_cnt | The number of all words contained in downloaded documents containing Terms of Service |
| Company's headquarter | hq | Country in which the top-parent company to the company providing the service is based in. If the company has no parent company, country in which the company providing the service is based in |
| Headquarter category | hq_cat | Higher level of "hq" variable, i.e. geographical area in which company's headquarter specified in "hq" is located: US; EU (excluding Poland); Poland; other |
| Publicly listed | public | Is the top-parent company to the company providing the service publicly listed, indirectly public (owned by a public company or a member of the public company's group) or private. If the company has no parent company, company providing the service was characterized. Public companies are those whose ownership structure is organized via shares or stock intended to be freely sold on the market. Indirectly public companies are those that are not public themselves, but which have been acquired by a publically listed company. Private companies are all the companies that do not belong to any of the other groups. In addition, information on the past public nature of the company is included ("previously public, delisted"), when it is applicable |
| Paid or free services | paid | Is the service paid, free to use, or optionally paid. Paid services require one-time or periodic payment for using them. Services with obligatory payment after short time of testing (e.g. 30 days; "trial" option) were also coded as paid. Free services require no payments for using them with full capabilities. Optionally paid services are free to use, but with limited functionality, and offer additional functionalities or facilities (e.g., more space, higher quality, unlimited use, additional options, no ads, etc.) against remuneration |

In [6]:
tos_eval_df['sector'].value_counts()

sector
Social           10
Games             8
Travel            8
Communication     7
Finance           7
Transport         7
Video             7
Food              6
Shopping          6
Sport             6
Various           6
Work              6
Cloud storage     5
Dating            5
Health            4
Music             2
Name: count, dtype: int64

### Exploring Annotated ToS Files

Function for extracting all comments from the annotated Word documents.

In [ ]:

NS = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}

def extract_comments_with_context(docx_path):
    with zipfile.ZipFile(docx_path) as z:
        # Load comments
        comments_xml = etree.fromstring(z.read('word/comments.xml'))
        comments = {
            c.get(f"{{{NS['w']}}}id"): {
                'author': c.get(f"{{{NS['w']}}}author", ""),
                'text': ''.join(c.xpath('.//w:t/text()', namespaces=NS))
            }
            for c in comments_xml.findall('.//w:comment', namespaces=NS)
        }
        
        # Load document content
        doc = etree.fromstring(z.read('word/document.xml'))
        
        results = []
        for cid, meta in comments.items():
            # XPath to collect all <w:t> between start and end markers for comment cid
            xpath = (
                f'//w:commentRangeStart[@w:id="{cid}"]'
                f'/following::w:t'
                f'[following::w:commentRangeEnd[@w:id="{cid}"]]'
            )
            texts = doc.xpath(xpath, namespaces=NS)
            referenced = ''.join(t.text or "" for t in texts)
            results.append({
                'id': cid,
                'author': meta['author'],
                'comment': meta['text'],
                'referenced_text': referenced
            })
        return results

# Usage
for item in extract_comments_with_context(f"{annotated_tos_path1}/adidas.docx"):
    print(f"💬 Comment #{item['id']} by {item['author']}")
    print("Referred text:", item['referenced_text'])
    print("Comment:", item['comment'])
    print("-" * 40)


💬 Comment #0 by Autor
Referred text: Privacy Policy 
Comment: docu 1
----------------------------------------
💬 Comment #1 by Autor
Referred text: Currently, Runtastic essentially oﬀers its users the following features and information in several languages:Platform.Personal proﬁle page, including personal details; News regarding Runtastic in short form;Mobile Health & Fitness Apps;Content, e.g. texts, pictures and videos, regarding sports, health and nutrition, that is presented by Runtastic and professional third parties (sports coaches, trainers,...);Status messages;A blog presenting company information, products and news regarding health and ﬁtness in long form; and Other content, such as:About us: Short description of Runtastic;Company oﬀers: Oﬀers of Runtastic addressed to companies; Advertising by Runtastic and/or third parties;Support for users;Privacy Policy and these T&C; Press & Media Center;Jobs; andLegal information.Apps. Applications for Apple iPhones, Android phones, and o

Extract comments and referred texts from ToS Annotator 1.

In [9]:
comments_dict = {}

for filename in os.listdir(annotated_tos_path1):
    if filename.endswith('.docx'):
        docx_path = os.path.join(annotated_tos_path1, filename)
        try:
            comments = extract_comments_with_context(docx_path)
            comments_dict[filename] = comments
        except Exception as e:
            print(f"Error processing {filename}: {e}")

for fname, comments in comments_dict.items():
    print(f"{fname}: {len(comments)} comments extracted")
    # print(f"Comments for {filename}:")
    # for comment in comments:
    #     print(f"💬 Comment #{comment['id']} by {comment['author']}")
    #     print("Referred text:", comment['referenced_text'])
    #     print("Comment:", comment['comment'])
    #     print("-" * 40)

Error processing ~$adidas.docx: [Errno 2] No such file or directory: './datasets/Annotated Terms of Service of 100 Online Platforms/Annotated ToS/Annotator 1/~$adidas.docx'
YouTube.docx: 17 comments extracted
SQUARE ENIX.docx: 53 comments extracted
Vinted.docx: 39 comments extracted
Dailymotion.docx: 18 comments extracted
MyTaxi.docx: 16 comments extracted
Pracuj.pl.docx: 26 comments extracted
Oktawave.docx: 20 comments extracted
FollowMyHealth.docx: 21 comments extracted
Spotify.docx: 56 comments extracted
Badoo.docx: 28 comments extracted
Allegro.docx: 34 comments extracted
Snapchat.docx: 43 comments extracted
Tinder.docx: 40 comments extracted
Viber.docx: 65 comments extracted
Twitch.docx: 40 comments extracted
MultiSport.docx: 21 comments extracted
EliteSingles.docx: 13 comments extracted
Uber.docx: 28 comments extracted
trivago.docx: 17 comments extracted
Bolt.docx: 28 comments extracted
Wolt.docx: 24 comments extracted
Ubisoft.docx: 41 comments extracted
vod.mdag.pl.docx: 23 comm

In [10]:
# Flatten the dictionary into a list of rows
rows = []
for filename, comments in comments_dict.items():
    for comment in comments:
        rows.append({
            'filename': filename,
            'comment_id': comment['id'],
            'author': comment['author'],
            'comment': comment['comment'],
            'referenced_text': comment['referenced_text']
        })

# Create DataFrame and export to CSV
comments_export_df = pd.DataFrame(rows)
comments_export_df.to_csv('annotated_tos_comments.csv', index=False)

Extract comments from ToS Annotator 2

In [37]:
comments_dict = {}

for filename in os.listdir(annotated_tos_path2):
    if filename.endswith('.docx'):
        docx_path = os.path.join(annotated_tos_path2, filename)
        print(f"Processing {filename}...")
        try:
            comments = extract_comments_with_context(docx_path)
            comments_dict[filename] = comments
        except Exception as e:
            print(f"Error processing {filename}: {e}")

for fname, comments in comments_dict.items():
    print(f"{fname}: {len(comments)} comments extracted")

Processing YouTube.docx...
Processing SQUARE ENIX.docx...
Processing Vinted.docx...
Processing Dailymotion.docx...
Processing MyTaxi.docx...
Processing Pracuj.pl.docx...
Processing Oktawave.docx...
Processing FollowMyHealth.docx...
Processing Spotify.docx...
Processing Badoo.docx...
Processing Allegro.docx...
Processing Snapchat.docx...
Processing Tinder.docx...
Processing Viber.docx...
Processing Twitch.docx...
Processing MultiSport.docx...
Processing EliteSingles.docx...
Processing Uber.docx...
Processing trivago.docx...
Processing Bolt.docx...
Processing Wolt.docx...
Processing Ubisoft.docx...
Processing vod.mdag.pl.docx...
Processing ZnanyLekarz.docx...
Processing iCloud.docx...
Processing HBO GO.docx...
Processing KAYAK.docx...
Processing PayPal.docx...
Processing DoorDash.docx...
Processing PayPay.docx...
Processing Discord.docx...
Processing Amazon.docx...
Processing Maya.docx...
Processing TikTok.docx...
Processing Lyft.docx...
Processing Sleep Cycle.docx...
Processing Crypto.c

In [38]:
# Flatten the dictionary into a list of rows
rows = []
for filename, comments in comments_dict.items():
    for comment in comments:
        rows.append({
            'filename': filename,
            'comment_id': comment['id'],
            'author': comment['author'],
            'comment': comment['comment'],
            'referenced_text': comment['referenced_text']
        })

# Create DataFrame and export to CSV
comments_export_df = pd.DataFrame(rows)
comments_export_df.to_csv('annotated_tos_comments_2.csv', index=False)

## Extracted Comments Data Preprocessing

In [14]:
tos_comments_df = pd.read_csv('annotated_tos_comments.csv')
tos_comments_df.head()

,filename,comment_id,author,comment,referenced_text
0,YouTube.docx,0,Autor,docu 1-3,Your use of the Service is subject to these te...
1,YouTube.docx,1,Autor,uncle,as permitted by applicable law;
2,YouTube.docx,2,Autor,docu 4,run contests on or through the Service that do...
3,YouTube.docx,3,Autor,serv_chg 0,YouTube is constantly changing and improving t...
4,YouTube.docx,4,Autor,uncle,operability issues. We’ll also provide you wit...


Remove `.docx` extension and rename column name.

In [18]:
tos_comments_df['filename'] = tos_comments_df['filename'].str.replace('.docx', '', regex=False)
tos_comments_df = tos_comments_df.rename(columns={'filename': 'company'})

How many comments per company ToS?

In [19]:
tos_comments_df['company'].value_counts()

company
ZipRecruiter                    76
Viber                           65
MyFitnessPal                    62
Baidu AI Cloud                  59
The Witcher - Monster Slayer    59
                                ..
Pyszne.pl                       12
Krakowskie Smaki                10
Sleep Cycle                      5
Zalando                          3
Telegram                         2
Name: count, Length: 99, dtype: int64

How many of each type of variable classification?

In [20]:
tos_comments_df['comment'].value_counts()

comment
uncle                                                                     146
what                                                                      126
contr_chg -1                                                               98
ltd -1                                                                     94
serv_chg -1                                                                87
                                                                         ... 
pato; infringing Art. 61 of the Polish Civil Code                           1
serv_chg -1; -1 as “reasons include, without limitation”                    1
which ones                                                                  1
pato 1; directly infringing Article 19(1)(c) of the Directive 2019/770      1
ltd _cap -1                                                                 1
Name: count, Length: 351, dtype: int64

Some comments have more than one evaluation in them (example below). Split up every grouped comment and create individual rows for each classification.

In [24]:
tos_comments_df[tos_comments_df['comment'] == 'cnt_modr 0’; disc -1; acc_sus 0; acc_del 0']

,company,comment_id,author,comment,referenced_text
2180,Epic Games,39,Autor,cnt_modr 0’; disc -1; acc_sus 0; acc_del 0,Epic may also at its sole discretion limit acc...


List all the possible variable classifications defined from Palka et al. (2023).

In [32]:
variable_classifications = {
    # Evaluative variables
    "ltd": "Limitation of company's liability",
    "ltd_cap": "Maximal threshold of company's liability", 
    "period": "Limitation period",
    "as_is": "No promises",
    "indemn": "Indemnification clause",
    "c_law": "Choice of law other than user's domicile",
    "c_forum": "Choice of forum other than user's domicile",
    "arb": "Mandatory arbitration",
    "class": "Class action waiver",
    "contr_chg": "Unilateral change of contract",
    "price_chg": "Unilateral change of future prices",
    "serv_chg": "Unilateral change of service by the company",
    "acc_del": "Account deletion and unilateral termination of contract by the company",
    "transfer": "Transfer of contractual rights to another subject",
    "cnt_del": "User content deletion",
    "acc_sus": "Account suspension",
    "recom": "Main parameters used in recommender systems",
    "com_sys": "Internal complaint-handling system",
    "cnt_retr": "Retrieval of digital content by the user",
    "IP": "Excessive user content IP license",
    "discret": "Discretional power to interpret the ToS",
    "interpret": "General interpretation clause",
    "sever": "Severability clauses",
    "suggest": "Right to incorporate user's feedback or suggestions without compensation",
    
    # Count variables
    "uncle": "Unclear law clause",
    "docu": "Other documents",
    
    # Pull-out text variables
    "core": "Promises and obligations",
    "what": "Description of the service",
    
    # Metadata
    "name": "Name of the service",
    "url": "ToS URL",
    "date": "Date effective",
    "lang": "ToS Language",
    "sector": "Service sector",
    "word_cnt": "Number of words",
    "hq": "Company's headquarter",
    "hq_cat": "Headquarter category",
    "public": "Publicly listed",
    "paid": "Paid or free services"
}

For rows with multiple classifications, split them by semicolon then create new rows. If the comment does not contain one of the above variable classifications, then remove it.

In [33]:
new_rows = []

for _, row in tos_comments_df.iterrows():
    classifications = [c.strip() for c in str(row['comment']).split(';') if c.strip()]
    for classification in classifications:
        if any(code in classification.lower() for code in variable_classifications.keys()):
            new_row = row.copy()
            new_row['comment'] = classification.lower()
            new_rows.append(new_row)

tos_comments_df = pd.DataFrame(new_rows)
tos_comments_df['comment'].value_counts()

comment
acc_del 0                                                                                                                  185
uncle                                                                                                                      147
acc_sus 0                                                                                                                  142
what                                                                                                                       129
ltd -1                                                                                                                     107
                                                                                                                          ... 
each party shall be entitled to transfer all or part of the contract to its affiliates without the consumer’s agreement      1
-1 as (5) allows the service provider to perform such actions arbitrarily                              

end of v1 (for now). trying out other datasets and topics first.